## First experiment - third configuration - size window 116.

In [ ]:
import sys
import os
import warnings
import csv
import glob
import matplotlib.backends.backend_pdf
import mne
import pandas as pd
from mne import Epochs, find_events, pick_events
from collections import OrderedDict
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from datetime import datetime
import numpy as np
from scipy import signal
import random

from analysis_tools import load_raw

from sklearn.pipeline import make_pipeline

from sklearn.model_selection import train_test_split

from mne.decoding import Vectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA

from sklearn.model_selection import cross_val_score, StratifiedShuffleSplit, StratifiedKFold

from pyriemann.estimation import ERPCovariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from pyriemann.spatialfilters import Xdawn
import pandas as pd
import seaborn as sns
from datetime import datetime as dt

from collections import OrderedDict
from sklearn import datasets

from sklearn.metrics import roc_auc_score

from sklearn.metrics import roc_auc_score, accuracy_score

from sklearn.model_selection import cross_val_predict

from sklearn.svm import LinearSVC
from sklearn.svm import SVC

from sklearn.metrics import confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import SGDClassifier

from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.svm import SVC

from sklearn.neighbors import KNeighborsClassifier

import random

### The generated statistics and authentication datasets will be saved in the "statistics" directory. If the directory does not exist, it is created.

In [ ]:
path = 'statistics'

if not os.path.exists(path):
    os.makedirs(path)

### Application of Notch to attenuate the frequency at 50 Hz, the sixth-order Butterworth band-pass filter with cut-off frequencies of 1-17 Hz, and ICA. After their application, the framework generates the epochs in Dataframe format.

In [ ]:
def process_by_subject(subject_name, plot_images):
    count = 1
    datasets = sorted(glob.glob('data/'+ subject_name + '_*.csv'))
    df_final = pd.DataFrame()
    array_epochs = []
    for dataset in datasets:
        sampling_rate = 256
        convert_raw_openbci_to_csv = False
        subject = 0
        session = 0
        ch_names = {}
        
        if convert_raw_openbci_to_csv:
            dataset = convert_new_openbci(dataset)
            
        raw = load_raw(dataset, sfreq=sampling_rate, stim_ind=8, replace_ch_names=None, ch_ind=[0, 1, 2, 3, 4, 5, 6, 7])

        for i, chn in enumerate(raw.ch_names):
            ch_names[chn] = i
                        
        sfreq = raw.info['sfreq']
        data_ini, times_ini = raw[:-1, int(sfreq * 1):int(sfreq * 10)]
        
        raw_notch = raw.copy().notch_filter([50.0])

        iir_params = dict(order=6, ftype='butter')
        raw_notch_and_filter = raw_notch.copy().filter(1, 17, method='iir', iir_params=iir_params)
        
        ica = mne.preprocessing.ICA(n_components=8, random_state=97)
        ica.fit(raw_notch_and_filter)
        
        raw_notch_and_filter_ica = raw_notch_and_filter.copy()
        
        ica.exclude = []
        eog_inds, eog_scores = ica.find_bads_eog(raw_notch_and_filter_ica, ['Fp1','Fp2'], threshold=1.5)
        ica.exclude = eog_inds
                
        ica.apply(raw_notch_and_filter_ica)

        events = find_events(raw_notch_and_filter_ica, shortest_event=1) 
        
        baseline = (None, 0)
        
        event_id = {'Target': 1, 'NoTarget': 2}
        reject = {'eeg': 100e-6}

        epochs = Epochs(raw_notch_and_filter_ica, events=events, event_id=event_id, tmin=-0.1, tmax=0.8, reject=reject, preload=True)
        epochs.pick_types(eeg=True)
    
        array_epochs.append(epochs)
        
        if count == 20:
            all_epochs = mne.concatenate_epochs(array_epochs, add_offset=True)
            df_final = all_epochs.to_data_frame()
            no_targets = np.count_nonzero(all_epochs.events[:, -1]==2)
        
            index_no_targets = []
            y = all_epochs.events[:, -1]

            while(no_targets != 0):
                position = random.randint(0, len(y)-1)
                if y[position] == 2 and position not in index_no_targets:
                    index_no_targets.append(position)
                    no_targets -= 1

            all_epochs.drop(index_no_targets)
            
            df_final_only_targets = all_epochs.to_data_frame()
            
            df_final.to_csv('statistics/df_{}.csv'.format(subject_name), index=False)
            df_final_only_targets.to_csv('statistics/df_{}_targets.csv'.format(subject_name), index=False)
        
        count = count + 1

In [ ]:
process_by_subject("Carlos", False)
process_by_subject("Javi", False)
process_by_subject("Kike", False)
process_by_subject("Leo", False)
process_by_subject("Lola", False)
process_by_subject("Mariaje", False)
process_by_subject("Mario", False)
process_by_subject("Marivi", False)
process_by_subject("Mati", False)
process_by_subject("Pedro", False)

### Getting the statistics using a sliding window size equal to 116.

In [ ]:
import numpy as np

def get_stadistical_values(channel, data):    
    dicc = dict()

    dicc[channel+"_Mean"] = np.mean(data[channel])
    dicc[channel+"_variance"] = np.var(data[channel])
    dicc[channel+"_deviation"] = np.std(data[channel])
    dicc[channel+"_max"] = np.max(data[channel])
    dicc[channel+"_summatory"] = np.sum(data[channel])
    dicc[channel+"_median"] = np.median(data[channel])
    
    dfReturned = pd.DataFrame()

    dfReturned = dfReturned.append(pd.DataFrame.from_dict(dicc, orient='index'))

    dfReturned = dfReturned.transpose()

    return dfReturned

In [ ]:
def aply_all_channels(workDF):

    channels = ["Fp1","Fp2","C3","C4","P7","P8","O1","O2"]
    
    window_size = 116

    allData = pd.DataFrame()

    for i in range(0, workDF.shape[0]):
        
        if ((i+window_size) > workDF.shape[0]):
            break

        vectors = workDF.copy().iloc[i:i+window_size]
        
        allChannels = pd.DataFrame()
        
        for channel in channels:
            aux = get_stadistical_values(channel, vectors)

            allChannels = pd.concat([allChannels, aux], axis=1)
            
        allChannels['Condition'] = 1
                
        allData = pd.concat([allChannels, allData], axis=0)
        
    return allData

In [ ]:
aply_all_channels(pd.read_csv('statistics/df_Carlos_targets.csv')).to_csv('statistics/df_statistics_Carlos_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Javi_targets.csv')).to_csv('statistics/df_statistics_Javi_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Kike_targets.csv')).to_csv('statistics/df_statistics_Kike_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Leo_targets.csv')).to_csv('statistics/df_statistics_Leo_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Lola_targets.csv')).to_csv('statistics/df_statistics_Lola_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Mariaje_targets.csv')).to_csv('statistics/df_statistics_Mariaje_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Mario_targets.csv')).to_csv('statistics/df_statistics_Mario_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Marivi_targets.csv')).to_csv('statistics/df_statistics_Marivi_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Mati_targets.csv')).to_csv('statistics/df_statistics_Mati_window_116.csv', index=False)
aply_all_channels(pd.read_csv('statistics/df_Pedro_targets.csv')).to_csv('statistics/df_statistics_Pedro_window_116.csv', index=False)

### Generation of five authentication datasets for each subject.

In [ ]:
def get_statistics_authentication(subject_name):    
    statistics_Carlos = pd.read_csv('statistics/df_statistics_Carlos_window_116.csv')
    statistics_Javi = pd.read_csv('statistics/df_statistics_Javi_window_116.csv')
    statistics_Kike = pd.read_csv('statistics/df_statistics_Kike_window_116.csv')
    statistics_Leo = pd.read_csv('statistics/df_statistics_Leo_window_116.csv')
    statistics_Lola = pd.read_csv('statistics/df_statistics_Lola_window_116.csv')
    statistics_Mariaje = pd.read_csv('statistics/df_statistics_Mariaje_window_116.csv')
    statistics_Mario = pd.read_csv('statistics/df_statistics_Mario_window_116.csv')
    statistics_Marivi = pd.read_csv('statistics/df_statistics_Marivi_window_116.csv')
    statistics_Mati = pd.read_csv('statistics/df_statistics_Mati_window_116.csv')
    statistics_Pedro = pd.read_csv('statistics/df_statistics_Pedro_window_116.csv')
    
    subjects = ["Carlos", "Javi", "Kike", "Leo", "Lola", "Mariaje", "Mario", "Marivi", "Mati", "Pedro"]
    statistics_subjects = [statistics_Carlos, statistics_Javi, statistics_Kike, statistics_Leo, statistics_Lola, statistics_Mariaje, statistics_Mario, statistics_Marivi, statistics_Mati, statistics_Pedro]
   
    new_statistics_targets_only = []
    
    index_subject = subjects.index(subject_name)
    subject = 0
        
    for statistic in statistics_subjects:
        if subject == index_subject:
            statistics_subject = statistic
            
        else:
            new_statistics_targets_only.append(statistic)
    
        subject += 1
    
    targets_subject = statistics_subject.shape[0]
    number_rest_subjects = len(new_statistics_targets_only)
    targets_rest_subjects = targets_subject // number_rest_subjects
    targets_last_subject = (targets_subject - (targets_rest_subjects * number_rest_subjects)) + targets_rest_subjects
    
    subject = 0
    for statistic in new_statistics_targets_only:     
        if(subject == number_rest_subjects - 1):
            targets_out = targets_last_subject

        else:
            targets_out = targets_rest_subjects
            
        rows_statistic = statistic.shape[0]
        index_selected_targets = []
               
        while(targets_out != 0):
            position = random.randint(0, rows_statistic-1)
            if position not in index_selected_targets:
                vector = statistic.copy().iloc[[position]]
                vector['Condition'] = 0
                statistics_subject = pd.concat([vector, statistics_subject], axis=0)
                statistics_subject = statistics_subject.reset_index(drop=True)
                index_selected_targets.append(position)
                targets_out -= 1
                
        subject += 1
    return statistics_subject
    
    
subjects = ["Carlos", "Javi", "Kike", "Leo", "Lola", "Mariaje", "Mario", "Marivi", "Mati", "Pedro"]

for subject in subjects:
    for i in range(5):
        get_statistics_authentication(subject).to_csv('statistics/statistics_{}_{}_first_experiment_third_configuration_window_116.csv'.format(subject, i))

### Generation of a CSV file that will contain the results obtained in the authentication process.

In [ ]:
header = ['Option/Classifier', 1, 2, 6, 7, 8, 9]
with open('first_experiment_third_configuration_window_116.csv', 'w', encoding='UTF8') as f:
    writer = csv.writer(f)
    
    writer.writerow(header)

### Authentication process using multiclass classification.

In [ ]:
clfs = OrderedDict()

clfs['Clasificador I'] = make_pipeline(Vectorizer(), StandardScaler(), LogisticRegression())
clfs['Clasificador II'] = make_pipeline(Vectorizer(), LDA(shrinkage='auto', solver='eigen'))
clfs['Clasificador VI'] = make_pipeline(Vectorizer(), RandomForestClassifier(random_state=42))
clfs['Clasificador VII'] = make_pipeline(Vectorizer(), QDA())
clfs['Clasificador VIII'] = make_pipeline(Vectorizer(), SVC(gamma='scale'))
clfs['Clasificador IX'] = make_pipeline(Vectorizer(), KNeighborsClassifier(n_neighbors=50))

def authentication_by_subject(subject_name, statistics, experiment):
   
    option = 'First_experiment_third_configuration_window_116_' + subject_name + '_' + str(experiment)
    
    data = []
    data.append(option)
    
    X = []
    y = []
        
    channels = statistics.loc[:, "Fp1_Mean":"O2_median"]
    X = channels.to_numpy()
    conditions = statistics.loc[:, "Condition"]
    y = conditions.to_numpy()
           
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    auc = []
    methods = []
    for m in clfs:
        clfs[m].fit(X_train, y_train)
        score = clfs[m].predict(X_test)
        report = classification_report(y_test, score, output_dict=True)
        accuracy = report['accuracy']
        accuracy = round(accuracy, 2)
        data.append(accuracy)
       
    with open('first_experiment_third_configuration_window_116.csv', 'a') as f:
        writer = csv.writer(f)
    
        writer.writerow(data)
        
        f.close()        

subjects = ["Carlos", "Javi", "Kike", "Leo", "Lola", "Mariaje", "Mario", "Marivi", "Mati", "Pedro"]

for subject in subjects:
    for i in range(5):
        authentication_by_subject(subject, pd.read_csv('statistics/statistics_{}_{}_first_experiment_third_configuration_window_116.csv'.format(subject, i)), i)